In [ ]:
# 03b — SHUTDOWN RESUME (run at home, NO retraining)
# PRE: notebook 03 đã chạy tới cell 10 (als_topn.json saved) và đã chạy CHECKPOINT cell
#      (model saved to Drive + als_state.json) trước khi tắt máy.
# POST: hoàn thành KILL-CONTRACT + persist (seed parquet, model_card) — đóng gói M2.
# KILL: thiếu als_state.json hoặc als_v1.0.0 trên Drive ⟹ phải re-run notebook 03 đầy đủ.

In [ ]:
# SETUP (identical to nb03 cell 1)
from google.colab import drive
drive.mount('/content/drive')
import os
BASE = '/content/drive/MyDrive/movielens32m'
CURATED = f'{BASE}/curated'; ARTIFACTS = f'{BASE}/artifacts'; EVID = f'{BASE}/evidence'; CKPT = f'{BASE}/checkpoints'
for d in [ARTIFACTS, EVID, CKPT]: os.makedirs(d, exist_ok=True)

import glob as _g
!dpkg -l | grep -q openjdk-11 || (apt-get update -qq && apt-get install -qq -y openjdk-11-jre-headless)
_jvm = sorted(_g.glob('/usr/lib/jvm/java-11*'))
assert _jvm, 'ERR: openjdk-11 not installed'
os.environ['JAVA_HOME'] = _jvm[0]
print('JAVA_HOME ->', os.environ['JAVA_HOME'])
!pip install -q pyspark==3.5.7

In [ ]:
# Session (identical to nb03 cell 2 — ratings kept cached: needed for gate + seed parquet)
import os, shutil
LOCAL = '/content/curated'
if not os.path.isdir(f'{LOCAL}/curated_ratings'):
    print('copying curated parquet Drive -> local disk (~500MB, 1-3 min)...')
    shutil.copytree(CURATED, LOCAL)

from pyspark.sql import SparkSession, functions as F
spark = (SparkSession.builder
         .master('local[*]')
         .appName('als-resume')
         .config('spark.driver.memory', '6g')
         .config('spark.sql.shuffle.partitions', '32')
         .getOrCreate())
spark.sparkContext.setLogLevel('ERROR')
spark.sparkContext.setCheckpointDir(CKPT)
ratings = spark.read.parquet(f'{LOCAL}/curated_ratings').cache()
n_total = ratings.count()
print(f'curated ratings: {n_total:,}')

In [ ]:
# Load checkpoint state + saved ALS model (NO retraining)
import json, pandas as pd
with open(f'{EVID}/als_state.json') as f:
    state = json.load(f)
print(json.dumps(state, indent=2))
assert int(state['n_total']) == n_total, 'KILL: curated data mismatch vs checkpoint'

from pyspark.ml.recommendation import ALSModel
MODEL_DIR = f'{BASE}/models/als_v1.0.0'
model = ALSModel.load(MODEL_DIR)
print(f'ALS model loaded: {MODEL_DIR}')
print(f'  user factors: {model.userFactors.count():,} | item factors: {model.itemFactors.count():,}')

In [ ]:
# Rebuild recs from als_topn.json (already on Drive) — cheaper than recommendForAllUsers again
docs = json.load(open(f'{ARTIFACTS}/als_topn.json'))
print(f'als_topn.json: {len(docs):,} user docs')

sample = docs[0]
assert set(sample.keys()) == {'userId', 'modelVersion', 'generatedAt', 'strategy', 'recommendations'}
assert sample['strategy'] == 'ALS' and len(sample['recommendations']) <= 10
assert all(r['rank'] == i + 1 for i, r in enumerate(sample['recommendations']))

rows = [(d['userId'], r['movieId'], r['score'], r['rank'])
        for d in docs for r in d['recommendations']]
recs = spark.createDataFrame(rows, ['userId', 'movieId', 'score', 'rank']).cache()
n_recs = recs.count()
print(f'recs rebuilt: {n_recs:,} rows (expect {len(docs) * 10:,} if every user has 10)')

In [ ]:
# GATE KILL-CONTRACT (full Spark check — same as nb03 cell 11, no already-rated leak)
rated_df = ratings.select('userId', 'movieId')
n_after_anti = recs.join(rated_df, ['userId', 'movieId'], 'left_anti').count()
assert n_recs == n_after_anti, f'KILL-CONTRACT: {n_recs - n_after_anti} already-rated items leaked into als_topn'
print(f'KILL-CONTRACT PASS: {n_recs:,} recommendations, 0 already-rated (full check, not sample)')
print('\n=== M2 ARTIFACTS READY: popular_movies.json, similar_movies.json, als_topn.json, metrics.csv ===')

In [ ]:
# PERSIST (same as nb03 cell 12 — seed parquet + model_card; model already on Drive)
import datetime
(ratings.select('userId', 'movieId', 'rating', 'rating_ts')
 .write.mode('overwrite').parquet(f'{ARTIFACTS}/user_history_seed.parquet'))
n_hist = n_total
print(f'user_history_seed.parquet: {n_hist:,} rows')

model_card = {
    'modelVersion': 'v1.0.0',
    'modelType': 'ALS (implicit feedback off, explicit ratings)',
    'config': {'rank': int(state['best']['rank']), 'regParam': float(state['best']['regParam']),
               'maxIter': 15, 'nonnegative': True, 'coldStartStrategy': 'drop', 'seed': 42},
    'metrics': {'rmse_val': float(state['best']['rmse_val']),
                'rmse_test': float(state['rmse_als_test']),
                'rmse_movemean_test': float(state['rmse_mm_test'])},
    'split': {'method': 'global_temporal_70_15_15',
              'cut_val': str(state['cut_val']), 'cut_test': str(state['cut_test'])},
    'artifacts': {'als_topn': f'{ARTIFACTS}/als_topn.json',
                  'popular_movies': f'{ARTIFACTS}/popular_movies.json',
                  'similar_movies': f'{ARTIFACTS}/similar_movies.json',
                  'user_history_seed': f'{ARTIFACTS}/user_history_seed.parquet',
                  'als_model_dir': MODEL_DIR},
    'generatedAt': datetime.datetime.utcnow().isoformat() + 'Z',
}
with open(f'{ARTIFACTS}/model_card.json', 'w') as f:
    json.dump(model_card, f, indent=2)
print(json.dumps(model_card['config'], indent=2))
print('\n=== FULL M2 PACKAGE: 4 artifacts + ALS model + model_card.json ===')